# 07 — BABILong Construction-Robustness Pilot

## Goal

Test whether the C1/C2 J-Lens retention pipeline behaves sensibly on BABILong
rather than only on the RULER construction.

RULER remains the primary RQ2 cohort.

This first stage performs no model forward pass. We first determine whether
BABILong `32k` actually places the relevant supporting fact outside the AHN
recent window of 32,640 tokens.


In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np
import json
import re

# Work whether Jupyter executes from repo root or notebooks/
cwd = Path.cwd()
if (cwd / "merged_ckpt").exists():
    REPO = cwd
elif (cwd.parent / "merged_ckpt").exists():
    REPO = cwd.parent
else:
    raise RuntimeError(f"Could not locate repo root from {cwd}")

MODEL = REPO / "merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN"
RESULT = REPO / "results/babilong/07_babilong_32k_inspection.json"

WINDOW = 32640
N = 20

tok = AutoTokenizer.from_pretrained(str(MODEL))

print("Repo:", REPO)
print("Model:", MODEL)
print("AHN recent window:", WINDOW)
print("Examples to inspect:", N)


Repo: /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks
Model: /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN
AHN recent window: 32640
Examples to inspect: 20


## 1 — Load BABILong 32k / qa1

In [2]:
ds = load_dataset("RMT-team/babilong", "32k")["qa1"]

print("rows:", len(ds))
print("columns:", ds.column_names)

print("\nExample 0")
print("question:", ds[0]["question"])
print("target:", repr(ds[0]["target"]))


Using the latest cached version of the dataset since RMT-team/babilong couldn't be found on the Hugging Face Hub


Found the latest cached dataset configuration '32k' at /workspace/.hf_home/datasets/RMT-team___babilong/32k/0.0.0/ee0d588794c7ac098062ee0d247c733d62e94fe2 (last modified on Wed Sep 16 15:49:00 2026).


rows: 100
columns: ['input', 'question', 'target']

Example 0
question: Where is Mary? 
target: 'bathroom'


## 2 — Inspect answer tokenization

BABILong has text answers rather than RULER's digit-sequence answers.

We inspect both raw and space-prefixed tokenization before adapting C1/C2.


In [3]:
targets = sorted(set(ds["target"]))

for target in targets:
    raw_ids = tok(
        target,
        add_special_tokens=False
    )["input_ids"]

    spaced_ids = tok(
        " " + target,
        add_special_tokens=False
    )["input_ids"]

    print(f"\nTarget: {target!r}")
    print("raw:       ", raw_ids, tok.convert_ids_to_tokens(raw_ids))
    print("with space:", spaced_ids, tok.convert_ids_to_tokens(spaced_ids))



Target: 'bathroom'
raw:        [65, 77832] ['b', 'athroom']
with space: [14852] ['Ġbathroom']

Target: 'bedroom'
raw:        [2721, 2966] ['bed', 'room']
with space: [13829] ['Ġbedroom']

Target: 'garden'
raw:        [70, 8341] ['g', 'arden']
with space: [13551] ['Ġgarden']

Target: 'hallway'
raw:        [42241, 3117] ['hall', 'way']
with space: [50802] ['Ġhallway']

Target: 'kitchen'
raw:        [74, 7454] ['k', 'itchen']
with space: [9780] ['Ġkitchen']

Target: 'office'
raw:        [26516] ['office']
with space: [5163] ['Ġoffice']


## 3 — Supporting-fact eviction inspection

For each example:

1. tokenize the complete context;
2. extract the queried person;
3. find a sentence containing both that person and the target location;
4. measure its distance from the end of the context;
5. classify it as evicted when distance > 32,640 tokens.

This is a dataset inspection, not yet the C1 experiment.


In [4]:
rows = []

for i in range(min(N, len(ds))):
    ex = ds[i]

    context_ids = tok(
        ex["input"],
        add_special_tokens=False
    )["input_ids"]

    match = re.search(
        r"Where is (.+?)\?",
        ex["question"]
    )

    person = match.group(1).strip() if match else None
    target = ex["target"].strip()

    sentences = re.split(
        r'(?<=[.!?])\s+',
        ex["input"]
    )

    matches = [
        sentence
        for sentence in sentences
        if person
        and person.lower() in sentence.lower()
        and target.lower() in sentence.lower()
    ]

    support = None
    distance = None

    if matches:
        support = matches[-1]

        char_pos = ex["input"].rfind(support)

        before_ids = tok(
            ex["input"][:char_pos],
            add_special_tokens=False
        )["input_ids"]

        distance = len(context_ids) - len(before_ids)

    row = {
        "id": i,
        "question": ex["question"],
        "target": target,
        "context_tokens": len(context_ids),
        "support_found": support is not None,
        "support": support,
        "support_distance_from_end": distance,
        "evicted_past_32640": bool(
            distance is not None and distance > WINDOW
        ),
    }

    rows.append(row)

    print(f"\n===== EXAMPLE {i} =====")
    print("question:", row["question"])
    print("target:", repr(row["target"]))
    print("context tokens:", row["context_tokens"])
    print("support found:", row["support_found"])

    if support is not None:
        print("support:", repr(support[:200]))
        print("distance from end:", distance)
        print("evicted past 32640:", row["evicted_past_32640"])



===== EXAMPLE 0 =====
question: Where is Mary? 
target: 'bathroom'
context tokens: 31226
support found: True
support: 'Mary journeyed to the bathroom.'
distance from end: 17288
evicted past 32640: False



===== EXAMPLE 1 =====
question: Where is Sandra? 
target: 'kitchen'
context tokens: 30650
support found: True
support: 'Sandra moved to the kitchen.'
distance from end: 1607
evicted past 32640: False



===== EXAMPLE 2 =====
question: Where is Mary? 
target: 'kitchen'
context tokens: 30824
support found: True
support: '"Swing to\nthe left, so that I may have a good chance." "No telling what\'ll come of it if you shoot." "I\'ll simply put a few holes through that canoe." Mary journeyed to the kitchen.'
distance from end: 2168
evicted past 32640: False



===== EXAMPLE 3 =====
question: Where is John? 
target: 'kitchen'
context tokens: 31129
support found: True
support: 'John went to the kitchen.'
distance from end: 4670
evicted past 32640: False



===== EXAMPLE 4 =====
question: Where is Sandra? 
target: 'bedroom'
context tokens: 30104
support found: True
support: 'Sandra moved to the bedroom.'
distance from end: 20021
evicted past 32640: False


...


===== EXAMPLE 14 =====
question: Where is Sandra? 
target: 'bathroom'
context tokens: 30391
support found: True
support: 'Sandra travelled to the bathroom.'
distance from end: 11779
evicted past 32640: False



===== EXAMPLE 15 =====
question: Where is Mary? 
target: 'hallway'
context tokens: 31517
support found: True
support: 'Mary moved to the hallway.'
distance from end: 8092
evicted past 32640: False



===== EXAMPLE 16 =====
question: Where is Sandra? 
target: 'kitchen'
context tokens: 31304
support found: True
support: 'But go on;\nlet us hear what followed." Sandra went to the kitchen.'
distance from end: 6194
evicted past 32640: False

===== EXAMPLE 17 =====
question: Where is Daniel? 
target: 'office'
context tokens: 30651
support found: True
support: 'Daniel journeyed to the office.'
distance from end: 28581
evicted past 32640: False



===== EXAMPLE 18 =====
question: Where is Sandra? 
target: 'kitchen'
context tokens: 30436
support found: True
support: 'Sandra journeyed to the kitchen.'
distance from end: 1281
evicted past 32640: False



===== EXAMPLE 19 =====
question: Where is John? 
target: 'bedroom'
context tokens: 31123
support found: True
support: '"I\ndaresay they\'re not all exhausted yet." "Perhaps," Hilary said slowly, "some\nplaces are like some people, the longer and\nbetter you know them, the more you keep\nfinding out in them to like." "Fathe'
distance from end: 645
evicted past 32640: False


## 4 — Summary / decision gate

If BABILong 32k produces too few genuinely evicted examples, the next inspection
will use BABILong `64k`.

C1/C2 will not be run until a valid evicted cohort is established.


In [5]:
lengths = [r["context_tokens"] for r in rows]

found = [r for r in rows if r["support_found"]]
evicted = [r for r in rows if r["evicted_past_32640"]]

summary = {
    "dataset": "RMT-team/babilong",
    "config": "32k",
    "task": "qa1",
    "n_inspected": len(rows),
    "ahn_recent_window": WINDOW,
    "context_tokens_min": int(np.min(lengths)),
    "context_tokens_median": float(np.median(lengths)),
    "context_tokens_max": int(np.max(lengths)),
    "support_found_n": len(found),
    "evicted_n": len(evicted),
    "evicted_fraction": len(evicted) / len(rows),
    "decision": (
        "32k_has_evicted_candidates"
        if len(evicted) > 0
        else "move_to_64k"
    ),
}

print(json.dumps(summary, indent=2))

RESULT.parent.mkdir(parents=True, exist_ok=True)

with RESULT.open("w") as f:
    json.dump(
        {
            "summary": summary,
            "rows": rows,
        },
        f,
        indent=2
    )

print("\nSaved:", RESULT)


{
  "dataset": "RMT-team/babilong",
  "config": "32k",
  "task": "qa1",
  "n_inspected": 20,
  "ahn_recent_window": 32640,
  "context_tokens_min": 30104,
  "context_tokens_median": 31021.5,
  "context_tokens_max": 31598,
  "support_found_n": 20,
  "evicted_n": 0,
  "evicted_fraction": 0.0,
  "decision": "move_to_64k"
}

Saved: /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks/results/babilong/07_babilong_32k_inspection.json


## 5 — BABILong 64k eviction inspection

The 32k cohort did not create an eviction regime:

- 20/20 supporting facts were found
- 0/20 examples exceeded the 32,640-token AHN recent window
- observed contexts were only about 30k–31.6k Qwen tokens

We therefore repeat the same construction check using BABILong `64k`.


In [6]:
from huggingface_hub import hf_hub_download

path64 = hf_hub_download(
    repo_id="RMT-team/babilong",
    filename="data/qa1/64k.json",
    repo_type="dataset",
)

ds64 = load_dataset(
    "json",
    data_files={"qa1": path64},
)["qa1"]

print("rows:", len(ds64))
print("columns:", ds64.column_names)


rows: 100
columns: ['input', 'question', 'target']


In [7]:
rows64 = []

for i in range(min(N, len(ds64))):
    ex = ds64[i]

    context_ids = tok(
        ex["input"],
        add_special_tokens=False
    )["input_ids"]

    match = re.search(
        r"Where is (.+?)\?",
        ex["question"]
    )

    person = match.group(1).strip() if match else None
    target = ex["target"].strip()

    sentences = re.split(
        r'(?<=[.!?])\s+',
        ex["input"]
    )

    matches = [
        sentence
        for sentence in sentences
        if person
        and person.lower() in sentence.lower()
        and target.lower() in sentence.lower()
    ]

    support = None
    distance = None

    if matches:
        support = matches[-1]

        char_pos = ex["input"].rfind(support)

        before_ids = tok(
            ex["input"][:char_pos],
            add_special_tokens=False
        )["input_ids"]

        distance = len(context_ids) - len(before_ids)

    row = {
        "id": i,
        "question": ex["question"],
        "target": target,
        "context_tokens": len(context_ids),
        "support_found": support is not None,
        "support": support,
        "support_distance_from_end": distance,
        "evicted_past_32640": bool(
            distance is not None and distance > WINDOW
        ),
    }

    rows64.append(row)

    print(f"\n===== 64K EXAMPLE {i} =====")
    print("question:", row["question"])
    print("target:", repr(row["target"]))
    print("context tokens:", row["context_tokens"])
    print("support found:", row["support_found"])

    if support is not None:
        print("distance from end:", distance)
        print("evicted past 32640:", row["evicted_past_32640"])



===== 64K EXAMPLE 0 =====
question: Where is Mary? 
target: 'bathroom'
context tokens: 63097
support found: True
distance from end: 37868
evicted past 32640: True



===== 64K EXAMPLE 1 =====
question: Where is Sandra? 
target: 'kitchen'
context tokens: 62694
support found: True
distance from end: 4127
evicted past 32640: False



===== 64K EXAMPLE 2 =====
question: Where is Mary? 
target: 'kitchen'
context tokens: 63361
support found: True
distance from end: 5152
evicted past 32640: False



===== 64K EXAMPLE 3 =====
question: Where is John? 
target: 'kitchen'
context tokens: 62870
support found: True
distance from end: 11072
evicted past 32640: False



===== 64K EXAMPLE 4 =====
question: Where is Sandra? 
target: 'bedroom'
context tokens: 60384
support found: True
distance from end: 45252
evicted past 32640: True


...


===== 64K EXAMPLE 15 =====
question: Where is Mary? 
target: 'hallway'
context tokens: 61610
support found: True
distance from end: 12898
evicted past 32640: False



===== 64K EXAMPLE 16 =====
question: Where is Sandra? 
target: 'kitchen'
context tokens: 62532
support found: True
distance from end: 8739
evicted past 32640: False



===== 64K EXAMPLE 17 =====
question: Where is Daniel? 
target: 'office'
context tokens: 63267
support found: True
distance from end: 58515
evicted past 32640: True



===== 64K EXAMPLE 18 =====
question: Where is Sandra? 
target: 'kitchen'
context tokens: 58653
support found: True
distance from end: 4696
evicted past 32640: False



===== 64K EXAMPLE 19 =====
question: Where is John? 
target: 'bedroom'
context tokens: 63220
support found: True
distance from end: 1096
evicted past 32640: False


In [8]:
lengths64 = [r["context_tokens"] for r in rows64]

found64 = [
    r for r in rows64
    if r["support_found"]
]

evicted64 = [
    r for r in rows64
    if r["evicted_past_32640"]
]

summary64 = {
    "dataset": "RMT-team/babilong",
    "config": "64k",
    "task": "qa1",
    "n_inspected": len(rows64),
    "ahn_recent_window": WINDOW,
    "context_tokens_min": int(np.min(lengths64)),
    "context_tokens_median": float(np.median(lengths64)),
    "context_tokens_max": int(np.max(lengths64)),
    "support_found_n": len(found64),
    "evicted_n": len(evicted64),
    "evicted_fraction": len(evicted64) / len(rows64),
    "decision": (
        "64k_valid_for_c1_pilot"
        if len(evicted64) > 0
        else "64k_still_no_eviction"
    ),
}

print(json.dumps(summary64, indent=2))

result64 = REPO / "results/babilong/07_babilong_64k_inspection.json"

with result64.open("w") as f:
    json.dump(
        {
            "summary": summary64,
            "rows": rows64
        },
        f,
        indent=2
    )

print("\nSaved:", result64)


{
  "dataset": "RMT-team/babilong",
  "config": "64k",
  "task": "qa1",
  "n_inspected": 20,
  "ahn_recent_window": 32640,
  "context_tokens_min": 58653,
  "context_tokens_median": 62667.0,
  "context_tokens_max": 63361,
  "support_found_n": 20,
  "evicted_n": 6,
  "evicted_fraction": 0.3,
  "decision": "64k_valid_for_c1_pilot"
}

Saved: /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks/results/babilong/07_babilong_64k_inspection.json


## 6 — Freeze the full 64k evicted cohort

The first 20 examples produced 6 genuinely evicted supporting facts.

Before spending GPU compute, scan all 100 `qa1` examples and save the IDs whose
supporting fact is more than 32,640 tokens from the end.

These IDs form the candidate cohort for the BABILong C1 pilot.


In [9]:
all64 = []

for i in range(len(ds64)):
    ex = ds64[i]

    context_ids = tok(
        ex["input"],
        add_special_tokens=False
    )["input_ids"]

    match = re.search(
        r"Where is (.+?)\?",
        ex["question"]
    )

    person = match.group(1).strip() if match else None
    target = ex["target"].strip()

    sentences = re.split(
        r'(?<=[.!?])\s+',
        ex["input"]
    )

    matches = [
        sentence
        for sentence in sentences
        if person
        and person.lower() in sentence.lower()
        and target.lower() in sentence.lower()
    ]

    support = matches[-1] if matches else None
    distance = None

    if support is not None:
        char_pos = ex["input"].rfind(support)

        before_ids = tok(
            ex["input"][:char_pos],
            add_special_tokens=False
        )["input_ids"]

        distance = len(context_ids) - len(before_ids)

    all64.append({
        "id": i,
        "question": ex["question"],
        "target": target,
        "context_tokens": len(context_ids),
        "support_found": support is not None,
        "support": support,
        "support_distance_from_end": distance,
        "evicted": bool(
            distance is not None
            and distance > WINDOW
        ),
    })

evicted_cohort = [
    r for r in all64
    if r["evicted"]
]

print("Total examples:", len(all64))
print("Support found:", sum(r["support_found"] for r in all64))
print("Evicted candidates:", len(evicted_cohort))
print(
    "Evicted fraction:",
    round(len(evicted_cohort) / len(all64), 3)
)

print("\nEvicted IDs:")
print([r["id"] for r in evicted_cohort])

print("\nFirst 10 candidates:")
for r in evicted_cohort[:10]:
    print(
        r["id"],
        repr(r["target"]),
        r["support_distance_from_end"]
    )


Total examples: 100
Support found: 100
Evicted candidates: 32
Evicted fraction: 0.32

Evicted IDs:
[0, 4, 6, 8, 11, 17, 21, 24, 29, 30, 31, 32, 37, 40, 41, 49, 50, 60, 66, 67, 68, 72, 73, 74, 77, 83, 84, 86, 88, 89, 90, 99]

First 10 candidates:
0 'bathroom' 37868
4 'bedroom' 45252
6 'garden' 34812
8 'kitchen' 57804
11 'office' 33666
17 'office' 58515
21 'bedroom' 48277
24 'bedroom' 34352
29 'garden' 42553
30 'garden' 33442


In [10]:
cohort_path = (
    REPO /
    "results/babilong/07_babilong_64k_evicted_cohort.json"
)

with cohort_path.open("w") as f:
    json.dump(
        {
            "dataset": "RMT-team/babilong",
            "config": "64k",
            "task": "qa1",
            "ahn_recent_window": WINDOW,
            "n_total": len(all64),
            "n_evicted": len(evicted_cohort),
            "candidate_ids": [
                r["id"]
                for r in evicted_cohort
            ],
            "rows": evicted_cohort,
        },
        f,
        indent=2
    )

print("Saved:", cohort_path)


Saved: /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks/results/babilong/07_babilong_64k_evicted_cohort.json
